In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [1]:
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
#from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
# 환경변수 로드
#load_dotenv()
# 모델 설정
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
llm = ChatOpenAI(model=LLM_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [5]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
#from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
#load_dotenv()
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
llm = ChatOpenAI(model=LLM_MODEL)
embeddings_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [6]:
documents = [
    {
        "doc_id": "doc_001",
        "title": "트랜스포머 아키텍처",
        "content": "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. "
                   "셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. "
                   "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    },
    {
        "doc_id": "doc_002",
        "title": "RAG 시스템 개요",
        "content": "RAG(Retrieval-Augmented Generation)는 검색과 생성을 결합한 기법입니다. "
                   "외부 지식 베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 활용하여 LLM이 답변을 생성합니다. "
                   "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있는 장점이 있습니다."
    },
    {
        "doc_id": "doc_003",
        "title": "벡터 임베딩과 유사도 검색",
        "content": "벡터 임베딩은 텍스트를 고차원 벡터 공간에 매핑하는 기술입니다. "
                   "코사인 유사도를 사용하여 의미적으로 유사한 문서를 검색합니다. "
                   "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 검색에 활용됩니다."
    },
    {
        "doc_id": "doc_004",
        "title": "프롬프트 엔지니어링",
        "content": "프롬프트 엔지니어링은 LLM에게 효과적인 지시를 설계하는 기술입니다. "
                   "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있습니다. "
                   "시스템 프롬프트와 사용자 프롬프트를 구분하여 역할과 지시를 명확히 합니다."
    },
    {
        "doc_id": "doc_005",
        "title": "파인튜닝과 전이학습",
        "content": "파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. "
                   "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. "
                   "전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다."
    },
    {
        "doc_id": "doc_006",
        "title": "토큰화와 텍스트 전처리",
        "content": "토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. "
                   "BPE(Byte Pair Encoding), WordPiece, SentencePiece 등의 알고리즘이 있습니다. "
                   "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "doc_id": "doc_007",
        "title": "LLM 평가 메트릭",
        "content": "LLM 평가에는 자동 메트릭과 인간 평가가 사용됩니다. "
                   "BLEU, ROUGE, BERTScore 등의 자동 메트릭은 참조 답변과의 유사도를 측정합니다. "
                   "LLM-as-Judge 방식은 다른 LLM을 활용하여 품질을 평가하는 최신 접근법입니다."
    },
    {
        "doc_id": "doc_008",
        "title": "청킹 전략",
        "content": "청킹은 긴 문서를 적절한 크기로 분할하는 전략입니다. "
                   "고정 크기 청킹, 의미 기반 청킹, 재귀적 청킹 등의 방법이 있습니다. "
                   "청크 크기와 오버랩은 검색 품질에 큰 영향을 미치며, 보통 500-1000 토큰을 사용합니다."
    },
    {
        "doc_id": "doc_009",
        "title": "하이브리드 검색",
        "content": "하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. "
                   "RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. "
                   "키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다."
    },
    {
        "doc_id": "doc_010",
        "title": "멀티모달 AI",
        "content": "멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. "
                   "GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. "
                   "CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    },
]


In [7]:
qa_dataset = [
    {
        "query_id": "q01",
        "question": "트랜스포머의 핵심 메커니즘은 무엇인가요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."
    },
    {
        "query_id": "q02",
        "question": "RAG 시스템의 장점은 무엇인가요?",
        "relevant_doc_ids": ["doc_002"],
        "ground_truth": "RAG는 환각을 줄이고 최신 정보를 반영할 수 있으며, 외부 지식 베이스를 활용하여 더 정확한 답변을 생성합니다."
    },
    {
        "query_id": "q03",
        "question": "벡터 유사도 검색에 사용되는 데이터베이스는?",
        "relevant_doc_ids": ["doc_003"],
        "ground_truth": "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 벡터 유사도 검색에 활용됩니다."
    },
    {
        "query_id": "q04",
        "question": "프롬프트 엔지니어링의 주요 기법은?",
        "relevant_doc_ids": ["doc_004"],
        "ground_truth": "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있으며, 시스템/사용자 프롬프트를 구분합니다."
    },
    {
        "query_id": "q05",
        "question": "효율적 파인튜닝 기법에는 어떤 것이 있나요?",
        "relevant_doc_ids": ["doc_005"],
        "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다."
    },
    {
        "query_id": "q06",
        "question": "한국어 토큰화에 적합한 방법은?",
        "relevant_doc_ids": ["doc_006"],
        "ground_truth": "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."
    },
    {
        "query_id": "q07",
        "question": "LLM-as-Judge란 무엇인가요?",
        "relevant_doc_ids": ["doc_007"],
        "ground_truth": "LLM-as-Judge는 다른 LLM을 활용하여 생성 품질을 평가하는 최신 접근법입니다."
    },
    {
        "query_id": "q08",
        "question": "적절한 청크 크기는 얼마인가요?",
        "relevant_doc_ids": ["doc_008"],
        "ground_truth": "보통 500-1000 토큰 크기를 사용하며, 청크 크기와 오버랩이 검색 품질에 큰 영향을 미칩니다."
    },
    {
        "query_id": "q09",
        "question": "하이브리드 검색에서 결과를 병합하는 방법은?",
        "relevant_doc_ids": ["doc_009"],
        "ground_truth": "RRF(Reciprocal Rank Fusion)를 통해 키워드 검색과 벡터 검색 결과를 효과적으로 병합합니다."
    },
    {
        "query_id": "q10",
        "question": "검색과 생성을 결합하여 환각을 줄이는 기법은?",
        "relevant_doc_ids": ["doc_002", "doc_003"],
        "ground_truth": "RAG(Retrieval-Augmented Generation)는 검색으로 관련 문서를 찾고 LLM이 이를 기반으로 답변을 생성하여 환각을 줄입니다."
    },
    {
        "query_id": "q11",
        "question": "CLIP 모델의 활용 분야는?",
        "relevant_doc_ids": ["doc_010"],
        "ground_truth": "CLIP은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."
    },
    {
        "query_id": "q12",
        "question": "트랜스포머의 구조는 어떻게 되어있나요?",
        "relevant_doc_ids": ["doc_001"],
        "ground_truth": "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."
    },
]

In [8]:
langchain_docs = [
    Document(page_content=doc["content"], metadata={"doc_id": doc["doc_id"], "title": doc["title"]})
    for doc in documents
]
vectorstore = FAISS.from_documents(langchain_docs, embeddings_model)
def search_documents(query: str, k: int = 5) -> List[Dict]:
    results = vectorstore.similarity_search_with_score(query, k=k)
    retrieved = []
    for doc, score in results:
        retrieved.append({
            "doc_id": doc.metadata["doc_id"],
            "title": doc.metadata["title"],
            "content": doc.page_content,
            "score": float(score)
        })
    return retrieved
search_results_cache = {}
for qa in qa_dataset:
    results = search_documents(qa["question"], k=5)
    search_results_cache[qa["query_id"]] = results

In [9]:
search_results_cache

{'q01': [{'doc_id': 'doc_001',
   'title': '트랜스포머 아키텍처',
   'content': "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. 셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. 인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다.",
   'score': 0.8629103899002075},
  {'doc_id': 'doc_010',
   'title': '멀티모달 AI',
   'content': '멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다.',
   'score': 1.3324825763702393},
  {'doc_id': 'doc_005',
   'title': '파인튜닝과 전이학습',
   'content': '파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. 전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다.',
   'score': 1.4281699657440186},
  {'doc_id': 'doc_009',
   'title': '하이브리드 검색',
   'content': '하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. 키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다.',
   'score': 1.5083041191101074},
  {'doc_id': 'doc_004',
   'title': '프롬프트 엔지니어링',
   

In [14]:
def precision_at_k(query_id, k):
  # query_id에 해당되는 값들을 가져와서
  qa = next(q for q in qa_dataset if q['query_id'] == query_id)
  relevant_ids = set(qa['relevant_doc_ids'])
  retrieved = search_results_cache[query_id][:k]
  retrieved_ids = {r['doc_id'] for r in retrieved} # retrieved에 있는 문서들을 for 루프를 돌면서 'doc_id'
                                          # 쿼리 아이디랑 유사하다고 판단된 문서의 아이디들이 들어있음
  relevant_retrieved = relevant_ids & retrieved_ids
  return len(relevant_retrieved) / k



## 리콜

In [10]:
def recall_at_k(query_id, k):
  qa = next(q for q in qa_dataset if q['query_id'] == query_id)
  relevant_ids = set(qa['relevant_doc_ids'])
  retrieved = search_results_cache[query_id][:k]
  retrieved_ids = {r['doc_id'] for r in retrieved} # retrieved에 있는 문서들을 for 루프를 돌면서 'doc_id'
                                          # 쿼리 아이디랑 유사하다고 판단된 문서의 아이디들이 들어있음
  relevant_retrieved = relevant_ids & retrieved_ids
  return len(relevant_retrieved) / len(relevant_ids) if relevant_ids else 0.0 # 에러처리,(있다면 len / len으로 계산)

In [13]:
def f1_at_k(query_id, k):
  p = precision_at_k(query_id, k)
  r = recall_at_k(query_id, k)

  if p+r ==0:
    return 0.0

  return 2*p*r / (p+r) # 조화평균

In [12]:
def r_precision(query_id):
  # 쿼리마다 문서개수가 달라도 공정하게 비교 가능
  qa = next(q for q in qa_dataset if q['query_id'] == query_id)
  relevant_ids = set(qa['relevant_doc_ids'])
  R = len(relevant_ids)

  if R == 0:
    return 0.0

  return precision_at_k(query_id, R)

In [26]:
def f_beta_at_k(query_id, k, beta):
  p = precision_at_k(query_id, k)
  r = recall_at_k(query_id, k)


  beta_sq = beta ** 2

  if beta_sq*p + r == 0:
    return 0.0

  return (1+beta_sq) * p * r / (beta_sq * p + r)

In [15]:
for k in [1, 3, 5]:
  avg_p = np.mean([precision_at_k(qa['query_id'], k) for qa in qa_dataset])
  avg_r = np.mean([recall_at_k(qa['query_id'], k) for qa in qa_dataset])
  avg_f1 = np.mean([f1_at_k(qa['query_id'], k) for qa in qa_dataset])

  print(f"K = {k}: P@K = {avg_p}, R@K = {avg_r}, F1@K = {avg_f1}")

K = 1: P@K = 1.0, R@K = 0.9583333333333334, F1@K = 0.9722222222222222
K = 3: P@K = 0.3333333333333333, R@K = 0.9583333333333334, F1@K = 0.4916666666666667
K = 5: P@K = 0.2166666666666667, R@K = 1.0, F1@K = 0.35317460317460325


In [17]:
avg_rp =  np.mean([r_precision(qa['query_id']) for qa in qa_dataset])

In [18]:
avg_rp

np.float64(0.9583333333333334)

In [20]:
for beta in [0.5, 1.0, 2.0]:
  avg_fb = np.mean([f_beta_at_k(qa['query_id'], 3, beta) for qa in qa_dataset])
  print(f"beta = {beta}, F{beta}@3 : {avg_fb}")

beta = 0.5, F0.5@3 : 0.3823260073260073
beta = 1.0, F1.0@3 : 0.4916666666666667
beta = 2.0, F2.0@3 : 0.6926406926406926


In [23]:
rows = []
for qa in qa_dataset:
    qid = qa['query_id']
    p = precision_at_k(qid, 3)
    r = recall_at_k(qid, 3)
    f1 = f1_at_k(qid, 3)
    rp = r_precision(qid)
    rows.append({
        'query_id': qid,
        'question': qa['question'][:30],
        'P@3': p,
        'R@3': r,
        'F1@3' :f1,
        'R-Precisions': rp
    })

pd.DataFrame(rows)

,query_id,question,P@3,R@3,F1@3,R-Precisions
0,q01,트랜스포머의 핵심 메커니즘은 무엇인가요?,0.333333,1.0,0.5,1.0
1,q02,RAG 시스템의 장점은 무엇인가요?,0.333333,1.0,0.5,1.0
2,q03,벡터 유사도 검색에 사용되는 데이터베이스는?,0.333333,1.0,0.5,1.0
3,q04,프롬프트 엔지니어링의 주요 기법은?,0.333333,1.0,0.5,1.0
4,q05,효율적 파인튜닝 기법에는 어떤 것이 있나요?,0.333333,1.0,0.5,1.0
5,q06,한국어 토큰화에 적합한 방법은?,0.333333,1.0,0.5,1.0
6,q07,LLM-as-Judge란 무엇인가요?,0.333333,1.0,0.5,1.0
7,q08,적절한 청크 크기는 얼마인가요?,0.333333,1.0,0.5,1.0
8,q09,하이브리드 검색에서 결과를 병합하는 방법은?,0.333333,1.0,0.5,1.0
9,q10,검색과 생성을 결합하여 환각을 줄이는 기법은?,0.333333,0.5,0.4,0.5


In [22]:
pd.DataFrame(rows)

""


In [ ]:
# k = 1~5까지 해보면서
# 평균 precision, 평균 recall, 평균 F-beta(beta = 0.5, 2.0)에 대한 메트릭을 구해볼것

In [33]:
results = {}

for k in range(1, 6):
    rows = []
    for qa in qa_dataset:
        qid = qa['query_id']
        rows.append({
            'query_id':   qid,
            'question':   qa['question'][:30],
            'Precision':  precision_at_k(qid, k),
            'Recall':     recall_at_k(qid, k),
            'F_beta_0.5': f_beta_at_k(qid, k, beta=0.5),
            'F_beta_2.0': f_beta_at_k(qid, k, beta=2.0),
        })

    df_k = pd.DataFrame(rows)
    means = df_k.drop(columns=['query_id', 'question']).mean()  # ← 수정
    results[k] = means
    print(f"=== K = {k} ===")
    print(means.to_string())
    print()

summary_df = pd.DataFrame(results).T
summary_df.index.name = 'k'
display(summary_df)

=== K = 1 ===
Precision     1.000000
Recall        0.958333
F_beta_0.5    0.986111
F_beta_2.0    0.962963

=== K = 2 ===
Precision     0.500000
Recall        0.958333
F_beta_0.5    0.550926
F_beta_2.0    0.805556

=== K = 3 ===
Precision     0.333333
Recall        0.958333
F_beta_0.5    0.382326
F_beta_2.0    0.692641

=== K = 4 ===
Precision     0.270833
Recall        1.000000
F_beta_0.5    0.315904
F_beta_2.0    0.642361

=== K = 5 ===
Precision     0.216667
Recall        1.000000
F_beta_0.5    0.256133
F_beta_2.0    0.573362



,Precision,Recall,F_beta_0.5,F_beta_2.0
k,,,,
1,1.000000,0.958333,0.986111,0.962963
2,0.500000,0.958333,0.550926,0.805556
3,0.333333,0.958333,0.382326,0.692641
4,0.270833,1.000000,0.315904,0.642361
5,0.216667,1.000000,0.256133,0.573362


In [34]:
k_range = list(range(1,6))
k_range

[1, 2, 3, 4, 5]

In [35]:
avg_p =  [np.mean([precision_at_k(qa['query_id'], k, beta) for qa in qa_dataset]) for k in k_range]
avg_r =  [np.mean([recall_at_k(qa['query_id'], k, beta) for qa in qa_dataset]) for k in k_range]
avg_beta_05 =  [np.mean([f_beta_at_k(qa['query_id'], k, beta) for qa in qa_dataset]) for k in k_range]
avg_p =  [np.mean([precision_at_k(qa['query_id'], k, beta) for qa in qa_dataset]) for k in k_range]

TypeError: precision_at_k() takes 2 positional arguments but 3 were given

## 생성에 대한 메트릭

In [ ]:
# 왜 전통 NLP 메트릭일까?
# API 호출 없고, 저렴하고, 재현 가능함
# 한계는 정답 텍스트 필요..

In [36]:
from collections import Counter
import numpy as np

In [39]:
def get_params(tokens, n):
  return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n +1 ))

In [38]:
# 나는 학교에 가고 있습니다.
# 0      1     2      3

In [40]:
def get_ngrams(tokens,n):
  return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

In [49]:
def bleu_score(reference, hypothesis, max_n):

    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    c, r = len(hyp_tokens), len(ref_tokens)
    bp = 1.0 if c >=r else np.exp(1- r/c) if c>0 else 0.0

    precision = {}
    for n in range(1, max_n+1):
        ref_ngrams = get_ngrams(ref_tokens, n)
        hyp_ngrams = get_ngrams(hyp_tokens, n)    # ''

        if len(hyp_ngrams) == 0:
            precision[n] = 0.0
            continue

        clipped = sum(min(cnt, ref_ngrams.get(ng, 0)) for ng, cnt in hyp_ngrams.items())
        precision[n] = clipped / sum(hyp_ngrams.values())

    log_avg, valid_n = 0.0, 0
    for n in range(1, max_n +1):
        if precision[n] > 0:
            log_avg += np.log(precision[n])
            valid_n += 1

    bleu = bp * np.exp(log_avg / valid_n) if valid_n >0 else 0.0

    return {'bleu' : round(bleu,4), 'bp' : round(bp, 4)}

In [42]:
ref = "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."
hyp = "트랜스포머는 셀프 어텐션 메커니즘을 사용하여 시퀀스 내 모든 위치 간 관계를 병렬로 처리합니다."

In [50]:
result = bleu_score(ref, hyp, 4)

In [51]:
result

{'bleu': np.float64(0.3955), 'bp': 1.0}

## LLM-as-judge
- LLM을 이용해서 평가
- Context relevance

In [59]:
def evaluate_context_relevance(question, contexts, model):
  """LLM을 사용하여 검색된 컨텍스트의 관련성 평가"""
  context_text = "\n\n".join([
      f"[컨텍스트 {i+1}]: {ctx}" for i, ctx in enumerate(contexts)
  ])

  prompt = f"""질문과 검색된 컨텍스트의 관련성을 평가하세요.

  # 질문
  {question}

  # 검색된 컨텍스트
  {context_text}

  # 평가 기준
  각 컨텍스트에 대해 1-5점으로 평가하세요:
  - 5 : 질문에 직접적으로 답할 수 있는 핵심 정보 포함
  - 4 : 질문과 매우 관련 있는 정보 포함
  - 3 : 부분적으로 관련 있는 정보 포함
  - 2 : 간접적으로만 관련 있음
  - 1 : 거의 관련 없음

  # 응답 형식 (JSON)
  {{"scores": [점수1, 점수2, ...], "average" : "평균 점수", "reasoning" : "평가 근거"}}

  반드시 유효환 JSON만 출력하세요

  """

  response = model.bind(response_format = {'type' : 'json_object'}).invoke([HumanMessage(content = prompt)])
  result = json.loads(response.content)

  return result


In [60]:
sample_qa = qa_dataset[0]

sample_qa

{'query_id': 'q01',
 'question': '트랜스포머의 핵심 메커니즘은 무엇인가요?',
 'relevant_doc_ids': ['doc_001'],
 'ground_truth': '트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다.'}

In [61]:
sample_results = search_results_cache[sample_qa['query_id']][:3]
sample_contexts = [r['content'] for r in sample_results]

cr_result = evaluate_context_relevance(sample_qa['question'], sample_contexts, llm)

In [62]:
sample_contexts

["트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. 셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. 인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다.",
 '멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다.',
 '파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. 전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다.']

In [64]:
cr_result

{'scores': [5, 1, 1],
 'average': '2.33',
 'reasoning': '컨텍스트 1은 트랜스포머의 핵심 메커니즘인 셀프 어텐션과 인코더-디코더 구조에 대해 직접 언급하고 있어 질문에 대해 가장 직접적인 답변을 제공합니다. 반면, 컨텍스트 2와 3은 트랜스포머와 직접적인 관련이 없으며 멀티모달 AI 및 파인튜닝에 대한 내용이므로 점수가 낮습니다.'}

In [57]:
search_results_cache['q01']

[{'doc_id': 'doc_001',
  'title': '트랜스포머 아키텍처',
  'content': "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. 셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. 인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다.",
  'score': 0.8629103899002075},
 {'doc_id': 'doc_010',
  'title': '멀티모달 AI',
  'content': '멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다.',
  'score': 1.3324825763702393},
 {'doc_id': 'doc_005',
  'title': '파인튜닝과 전이학습',
  'content': '파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. 전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다.',
  'score': 1.4281699657440186},
 {'doc_id': 'doc_009',
  'title': '하이브리드 검색',
  'content': '하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. 키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다.',
  'score': 1.5083041191101074},
 {'doc_id': 'doc_004',
  'title': '프롬프트 엔지니어링',
  'content': '프롬프트 엔지니어링은 LL

In [65]:
for i, (r, score) in enumerate(zip(sample_results, cr_result.get('scores', []))):

  print(f"컨텍스트 {i+1} ({r['doc_id']}), {score}/5")

컨텍스트 1 (doc_001), 5/5
컨텍스트 2 (doc_010), 1/5
컨텍스트 3 (doc_005), 1/5
